In [1]:
# pip install scikit-learn

# Feature selection for US-accident dataset

## Importing required Libraries

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor , XGBClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report,mean_squared_error,mean_absolute_error,r2_score

## Dataset loading

In [3]:
df = pd.read_csv('/kaggle/input/datasets/adityamishra0156/us-accident-processed/Dataset.csv')

In [4]:
'''df['Severity_bin'] = df['Severity'].map({
    1: 0,
    2: 0,
    3: 1,
    4: 1
})'''

"df['Severity_bin'] = df['Severity'].map({\n    1: 0,\n    2: 0,\n    3: 1,\n    4: 1\n})"

In [5]:
df = df.astype('float32')

## Severity feature Analysis

In [6]:
df['Severity'] = df['Severity']-1

In [7]:
print(df['Severity'].isna().sum())
# print(df['Severity_bin'].isna().sum())
df['Severity'].value_counts()

0


Severity
1.0    6156981
2.0    1299337
3.0     204710
0.0      67366
Name: count, dtype: int64

In [8]:
X = df.drop(columns = ['Severity','Distance(mi)','Accident_duration_hr'])
y =df['Severity']

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.8,random_state=42)

In [11]:
X_train.fillna(X_train.mean(), inplace=True)
X_test = X_test.fillna(X_test.mean())

In [12]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    learning_rate=0.1,
    class_weight='balanced',
    random_state=42
)

lgbm.fit(X_train, y_train)
y_pred = lgbm.predict(X_test)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.703528 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1806
[LightGBM] [Info] Number of data points in the train set: 6182715, number of used features: 58
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294


In [13]:
print('Lightgbm')
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, average='macro'))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))

print(classification_report(y_test, y_pred))

Lightgbm
Accuracy: 0.46073279121991045
F1: 0.31342694105182833
Precision: 0.7959669982138209
Recall: 0.46073279121991045
              precision    recall  f1-score   support

         0.0       0.05      0.86      0.09     13509
         1.0       0.92      0.43      0.59   1230523
         2.0       0.36      0.57      0.44    260525
         3.0       0.08      0.59      0.14     41122

    accuracy                           0.46   1545679
   macro avg       0.35      0.61      0.31   1545679
weighted avg       0.80      0.46      0.54   1545679



In [14]:
from catboost import CatBoostClassifier

cb = CatBoostClassifier(
    learning_rate=0.1,
    auto_class_weights='Balanced',
    verbose=0
)

cb.fit(X_train, y_train)
y_pred = cb.predict(X_test)

In [15]:
print('Catboost')
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, average='macro'))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))

print(classification_report(y_test, y_pred))

Catboost
Accuracy: 0.48584926106908355
F1: 0.3281274527920962
Precision: 0.8022647610480406
Recall: 0.48584926106908355
              precision    recall  f1-score   support

         0.0       0.05      0.87      0.10     13509
         1.0       0.93      0.45      0.61   1230523
         2.0       0.37      0.60      0.46    260525
         3.0       0.09      0.61      0.15     41122

    accuracy                           0.49   1545679
   macro avg       0.36      0.63      0.33   1545679
weighted avg       0.80      0.49      0.57   1545679



In [16]:
xgbc = XGBClassifier(
    learning_rate=0.1
) # mandatory

xgbc.fit(X_train,y_train)
y_pred = xgbc.predict(X_test.values.astype("float32"))

In [17]:
print(xgbc.score(X_train, y_train))
print(xgbc.score(X_test, y_test))

0.8019310286823831
0.8011637603926818


In [18]:
print('XGBoost')
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, average='macro'))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))

print(classification_report(y_test, y_pred))

XGBoost
Accuracy: 0.8011637603926818
F1: 0.26527669015859634
Precision: 0.7729815117623242
Recall: 0.8011637603926818
              precision    recall  f1-score   support

         0.0       0.53      0.04      0.07     13509
         1.0       0.80      1.00      0.89   1230523
         2.0       0.69      0.05      0.09    260525
         3.0       0.51      0.00      0.01     41122

    accuracy                           0.80   1545679
   macro avg       0.63      0.27      0.27   1545679
weighted avg       0.77      0.80      0.72   1545679



In [19]:
result = permutation_importance(
    cb,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)


In [20]:
importance = pd.Series(result.importances_mean, index=X_test.columns)
importance = importance.sort_values(ascending=False)

In [21]:
important_features_severity = importance.head(25).index.tolist()

In [22]:
important_features_severity 

['Wind_Speed(mph)',
 'County_Log_Count',
 'Pressure(in)',
 'Traffic_Signal',
 'Temperature',
 'City_Log_Count',
 'Humidity(%)',
 'Month_September',
 'Stop',
 'Junction',
 'Month_October',
 'is_windy',
 'Period_Morning',
 'Weekday_Sunday',
 'is_clear',
 'Period_Evening',
 'Weekday_Saturday',
 'Period_Early Morning',
 'Sunrise_Sunset_Unknown',
 'Precipitation(in)',
 'is_snow',
 'Station',
 'Give_Way',
 'is_cloud',
 'Weekday_Monday']

## Distance feature Analysis

In [23]:
y_d = df['Distance(mi)']

In [24]:
X = df.drop(columns=['Distance(mi)','Accident_duration_hr'])

In [25]:
X_train,X_test,y_train_d,y_test_d = train_test_split(X,y_d,train_size=0.8,random_state=42)

In [26]:
print(y_train_d.quantile([0.5, 0.9, 0.95, 0.99]))

0.50    0.030
0.90    1.469
0.95    2.670
0.99    6.947
Name: Distance(mi), dtype: float64


In [27]:
y_train_d = np.log1p(y_train_d)
y_test_d  = np.log1p(y_test_d)

In [28]:
X_train.fillna(X_train.mean(),inplace=True)

In [29]:
X_test = X_test.fillna(X_test.mean())

In [30]:
xgbr = XGBRegressor(learning_rate=0.1)

In [31]:
xgbr.fit(X_train,y_train_d)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [32]:
y_pred_d = xgbr.predict(X_test)

In [33]:
mae = mean_absolute_error(y_test_d,y_pred_d)
mse = mean_squared_error(y_test_d,y_pred_d)
r2 = r2_score(y_test_d,y_pred_d)

print('MAE : ',mae)
print('MSE : ',mse)
print('R2_score : ',r2)

MAE :  0.2847118079662323
MSE :  0.183017760515213
R2_score :  0.16481506824493408


In [34]:
lr = LinearRegression()

In [35]:
lr.fit(X_train,y_train_d)

LinearRegression()

In [36]:
y_pred_d = lr.predict(X_test)
mae = mean_absolute_error(y_test_d,y_pred_d)
mse = mean_squared_error(y_test_d,y_pred_d)
r2 = r2_score(y_test_d,y_pred_d)

print('MAE : ',mae)
print('MSE : ',mse)
print('R2_score : ',r2)

MAE :  0.33430108428001404
MSE :  0.21897824108600616
R2_score :  0.0007126927375793457


In [37]:
result = permutation_importance(
    xgbr,
    X_test,
    y_test_d,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)


In [38]:
importance = pd.Series(result.importances_mean, index=X_test.columns)
importance = importance.sort_values(ascending=False)

In [39]:
important_features_distance = importance.head(25).index.tolist()

In [40]:
important_features_distance 

['Severity',
 'Traffic_Signal',
 'County_Log_Count',
 'City_Log_Count',
 'Crossing',
 'Pressure(in)',
 'Wind_Speed(mph)',
 'Junction',
 'Temperature',
 'Stop',
 'Period_Afternoon',
 'Station',
 'Visibility(mi)',
 'Amenity',
 'Humidity(%)',
 'Period_Morning',
 'is_snow',
 'is_severe_weather',
 'Weekday_Saturday',
 'Sunrise_Sunset_Unknown',
 'Period_Early Morning',
 'is_windy',
 'Weekday_Sunday',
 'Period_Night',
 'Period_Evening']

## Duration feature Analysis

In [41]:
y_t = df['Accident_duration_hr']

In [42]:
X = df.drop(columns=['Accident_duration_hr'])

In [43]:
X_train, X_test, y_train_t, y_test_t = train_test_split(X,y_t, train_size=0.8, random_state=42)

In [44]:
xgbr_t_normal = XGBRegressor(random_state=42)
xgbr_t_normal.fit(X_train, y_train_t)
y_pred_t = xgbr_t_normal.predict(X_test)

print("Without log")
print("MAE :", mean_absolute_error(y_test_t, y_pred_t))
print("MSE :", mean_squared_error(y_test_t, y_pred_t))
print("R2_score :", r2_score(y_test_t, y_pred_t))


Without log
MAE : 14.43275260925293
MSE : 46312.859375
R2_score : 0.16555076837539673


In [45]:
result = permutation_importance(
    xgbr_t_normal,
    X_test,
    y_test_t,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)


In [46]:
importance = pd.Series(result.importances_mean, index=X_test.columns)
importance = importance.sort_values(ascending=False)

In [47]:
important_features_duration_xgb = importance.head(25).index.tolist()

In [48]:
important_features_duration_xgb 

['County_Log_Count',
 'City_Log_Count',
 'Temperature',
 'Pressure(in)',
 'Distance(mi)',
 'Humidity(%)',
 'Severity',
 'Month_April',
 'Month_March',
 'Junction',
 'Wind_Speed(mph)',
 'Month_February',
 'Month_October',
 'Month_December',
 'Weekday_Saturday',
 'Period_Evening',
 'Period_Night',
 'Weekday_Tuesday',
 'Month_May',
 'Sunrise_Sunset_Unknown',
 'Weekday_Thursday',
 'Visibility(mi)',
 'Precipitation(in)',
 'Weekday_Wednesday',
 'Crossing']

In [49]:
rf = RandomForestRegressor(n_estimators=100,      # number of trees (keep moderate)
    max_depth=10,          # limit depth → faster + less overfitting
    min_samples_split=10,  # avoid too many splits
    min_samples_leaf=5,    # smoother trees
    max_features='sqrt',   # faster than using all features
    n_jobs=-1,             # use all CPU cores
    random_state=42)

In [50]:
rf.fit(X_train,y_train_t)

RandomForestRegressor(max_depth=10, max_features='sqrt', min_samples_leaf=5,
                      min_samples_split=10, n_jobs=-1, random_state=42)

In [51]:
y_pred_t = rf.predict(X_test)

In [52]:
print("MAE :", mean_absolute_error(y_test_t, y_pred_t))
print("MSE :", mean_squared_error(y_test_t, y_pred_t))
print("R2_score :", r2_score(y_test_t, y_pred_t))

MAE : 11.29948732450041
MSE : 53276.944156321224
R2_score : 0.04007435327000708


In [53]:
feature_scores = pd.DataFrame({
    'Feature_Name': X.columns,
    'Importance_Score': rf.feature_importances_
})

feature_scores = feature_scores.sort_values(by='Importance_Score', ascending=False)

important_features_duration_rf = feature_scores.head(25)['Feature_Name'].tolist()
print(important_features_duration_rf)

['County_Log_Count', 'City_Log_Count', 'Distance(mi)', 'Pressure(in)', 'Temperature', 'Humidity(%)', 'Wind_Speed(mph)', 'Severity', 'Visibility(mi)', 'Junction', 'Sunrise_Sunset_Unknown', 'Month_February', 'Period_Morning', 'is_clear', 'is_cloud', 'Precipitation(in)', 'Period_Night', 'Sunrise_Sunset_Night', 'Weekday_Saturday', 'Sunrise_Sunset_Day', 'Weekday_Thursday', 'Weekday_Sunday', 'Weekday_Friday', 'Month_November', 'Weekday_Tuesday']


In [54]:
features_duration = list(set(important_features_duration_rf).intersection(set(important_features_duration_xgb)))

## Final

#### common features for all

In [55]:
print("=== Severity features ===")
print(important_features_severity)

print("\n=== Distance features ===")
print(important_features_distance)

print("\n=== Duration features ===")
print(features_duration)

print("\n=== Common across all three ===")
common = set(important_features_severity) & set(important_features_distance) & set(features_duration)
print(sorted(common))

=== Severity features ===
['Wind_Speed(mph)', 'County_Log_Count', 'Pressure(in)', 'Traffic_Signal', 'Temperature', 'City_Log_Count', 'Humidity(%)', 'Month_September', 'Stop', 'Junction', 'Month_October', 'is_windy', 'Period_Morning', 'Weekday_Sunday', 'is_clear', 'Period_Evening', 'Weekday_Saturday', 'Period_Early Morning', 'Sunrise_Sunset_Unknown', 'Precipitation(in)', 'is_snow', 'Station', 'Give_Way', 'is_cloud', 'Weekday_Monday']

=== Distance features ===
['Severity', 'Traffic_Signal', 'County_Log_Count', 'City_Log_Count', 'Crossing', 'Pressure(in)', 'Wind_Speed(mph)', 'Junction', 'Temperature', 'Stop', 'Period_Afternoon', 'Station', 'Visibility(mi)', 'Amenity', 'Humidity(%)', 'Period_Morning', 'is_snow', 'is_severe_weather', 'Weekday_Saturday', 'Sunrise_Sunset_Unknown', 'Period_Early Morning', 'is_windy', 'Weekday_Sunday', 'Period_Night', 'Period_Evening']

=== Duration features ===
['City_Log_Count', 'Severity', 'Weekday_Tuesday', 'Wind_Speed(mph)', 'Pressure(in)', 'Visibility(mi